# Point Net Experimental

In this notebook I will explore the [PointNet](https://arxiv.org/abs/1612.00593) architecture in order to understand how it works. It seems for classification, the model ```PointNetCls``` and for segmentation the model ```PointNetDenseCls``` is used. There are other variants of models (```PointNetfeat```, ```STNkd```, ```STN3d```). As it is the simpler model, we will start with the classification model.

## PointNetCls

In [1]:
import torch

In [2]:
batchSize = 32
num_points = 2500
workers = 4
nepoch = 250
outf  = 'cls' # output folder
model = ''
dataset = '' # dataset path
dataset_type = 'shapenet'
feature_transform = True

In [3]:
blue = lambda x: '\033[94m' + x + '\033[0m'
print(blue("Lol das ist ja in blau!")) 

Lol das ist ja in blau!


In [4]:
manualSeed = 42
torch.manual_seed(manualSeed)

After setting preliminary stuff, first a dataset has to be created using the ```ShapeNetDataset``` class. It will be implemented here

START WITH NEW GITHUB REPO Pointnet_Pointnet2_pytorch which contains both pointnet models but in pytorch!!
Especially look out for the final layers and the how to change the loss to regression.


# PointNet from Pointnet_Pointnet2

Starting with the new repo, we will go through PointNet in order to understand how the latent representation z can be reconstructed. Therefore we will scrutinize the ```train_classification.py``` file using the ```pointnet_cls.py``` model.

The classification script works as follows:
- Check Cuda devices
- Create experiment logging directory
- Create a logger for detailed logging info
- Load the data -> data output format is: point_set, target_class_label -> **We have to adapt the dataloader** so output is: point_set, latent_vector
- Then the model and loss are loaded, also the python files necessary for training are copied into the experiment directory -> **Here we have to change the loss to MSE**
- Then the script checks for checkpoints and initializes Adam optimizer and learning rate scheduler

#### Training
- Before being processed, the points are randomly droped out, scaled and shifted (Why?)
- Then they are processed by the Classifier
- The ```PointNetEncoder``` produces the global feature vector and contains the important model architecture -> **One can take this encoder and use it for DeepCAD**
- First the encoder predicts a matrix for affine transformation of the point features (T-Net) -> This transformation matrix is part of the loss calculation, where the loss tries to keep the transformation matrices as orthogonal as possible ($A \cdot A^T = I$)
- Then a per point MLP/1D Convolution per point (same as MLP) is applied to increase feature space and another transformation matrix is predicted -> Actually only this one goes into the loss, not the point features
- Finally you get the global feature vector

#### Next steps
- Understand pointnet++ (or try first with simpler pointnet?)
- Change loss to MSE regression loss
- Change data to: Input->PC, Target -> Latent vector
- Find a good workflow, i.e. Should I just copy the important code parts from the github repo or should I fork and change some things? What are best practices?

# PointNet++ from Pointnet_Pointnet2

After understanding how PointNet works, let's scrutinize its successor PointNet++. In contrast to its predecessor, PointNet++ uses a hierarchical structure to obtain the global feature vector, thereby putting more emphasis on local features. The setup script has the same structure as in PointNet which is why we will start by looking at the training.

#### Training 1
- The script first checks if the points contain normals or not. In our case we will have no normals. The points are sent to the first _set abstraction_ layer
- Then centroid points need to be sampled using iterative farthest point sampling (FPS) -> This algorithm basically always chooses the point which is the furthest away from a set of points and results in a good coverage of the point cloud with centroid points -> Result: For each batch sample you get a specific number of centroid points with their index
- Then the script retrieves the points behind the sampled centroid indices
- Then grouping starts
- Maybe implement this part? Or watch a video? Currently I know what the code does, but I do not understand it 100%, but maybe thats enough for now
- **Continue with grouping**

#### Farthest Point sampling
The algorithm is quite simple. You start with a point cloud comprising N points and iteratively select a point until you have up to S samples. You have two sets which we will denote sampled and remaining and you choose a point as follows:
- For each point in remaining find its nearest neighbour in sampled, saving the distance.
- Select the point in remaining whose nearest neighbour distance is the largest and move it from remaining to sampled.

#### Training 2
- 

In [54]:
x = torch.eye(2)
x[None,:].shape

torch.Size([2, 2])

In [21]:
x.transpose(2,1)
x.shape

torch.Size([2, 3, 4])

In [30]:
a,b,c= x.size()
a, b

(2, 3)

In [32]:
x = torch.ones(10, 3, 100)
batchsize = x.size()[0]

In [34]:
conv = torch.nn.Conv1d(3, 64, 1)

In [56]:
def lol(a,b):
    if b is None:
        print("HAHA")
    return a+1

lol(3)

TypeError: lol() missing 1 required positional argument: 'b'

In [36]:
a.shape

torch.Size([10, 64, 100])

In [43]:
x = torch.Tensor([[1,2,3],[4,5,6]])

In [44]:
x

tensor([[1., 2., 3.],
        [4., 5., 6.]])

In [49]:
x.transpose(1,0)

tensor([[1., 4.],
        [2., 5.],
        [3., 6.]])